# KisanMitra ML Practical

Simple crop recommendation workflow. Run from the project folder after adding `data/Crop_recommendation.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 1. Load and inspect the dataset

In [ ]:
df = pd.read_csv("../data/Crop_recommendation.csv")
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

## 2. Clean the data and view crop counts

In [ ]:
df.columns = df.columns.str.strip()
df = df.drop_duplicates()
df = df.dropna()
features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
df = df[features + ["label"]]
df.describe()

In [ ]:
df["label"].value_counts().plot(kind="bar", figsize=(10, 4))
plt.title("Number of records for each crop")
plt.tight_layout()
plt.show()

## 3. Split the data

In [ ]:
X = df[features]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 4. Train and tune four models

In [ ]:
models = {
    "KNN": (make_pipeline(StandardScaler(), KNeighborsClassifier()), {"kneighborsclassifier__n_neighbors": [3, 5, 7]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=42), {"max_depth": [5, 10, None]}),
    "SVM": (make_pipeline(StandardScaler(), SVC()), {"svc__C": [1, 5], "svc__kernel": ["linear", "rbf"]}),
    "Random Forest": (RandomForestClassifier(random_state=42), {"n_estimators": [100], "max_depth": [10, None]})
}
results = []
best_model = None
best_cv_score = -1
best_name = ""


In [ ]:
for name, (model, params) in models.items():
    search = GridSearchCV(model, params, cv=5, scoring="f1_macro", n_jobs=-1)
    search.fit(X_train, y_train)
    pred = search.predict(X_test)
    print("\n", name, search.best_params_)
    print(classification_report(y_test, pred, zero_division=0))
    results.append([name, accuracy_score(y_test, pred), precision_score(y_test, pred, average="macro", zero_division=0), recall_score(y_test, pred, average="macro", zero_division=0), f1_score(y_test, pred, average="macro", zero_division=0), search.best_score_])
    if search.best_score_ > best_cv_score:
        best_cv_score = search.best_score_
        best_model = search.best_estimator_
        best_name = name


## 5. Compare model performance

In [ ]:
results = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1", "CV F1"])
results.sort_values("F1", ascending=False)

In [ ]:
results.set_index("Model")["F1"].plot(kind="bar", title="Model comparison")
plt.ylabel("Macro F1 score")
plt.tight_layout()
plt.show()
print("Best model selected using training cross-validation:", best_name)